<a href="https://colab.research.google.com/github/XiayuZhou0524/analytic-engineering/blob/main/2_03_practice_bike_hire_student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![](https://drive.google.com/uc?export=view&id=17M9oGaOj4KdKiDDH3oTibxJ9qBTSjp6n)

# 2.03 Practice session
## Bike-hire trips

## How this practice works

This is a **short rehearsal** of the timed test. You do not need to finish or get the "right" answer. The aim is to practise the way of working.

- You have about **45 minutes**. Work individually.
- You may use **any AI tool** (for example Gemini in Colab). We are **not marking** anything today. In the real test, the quality of AI-generated code will not matter. What matters is the control youDshow.
- Write in markdown cells as you go: your thinking, **each prompt you send (paste it as you send it)**, and the checks you make.
- Unfinished work is fine. Say what you did not get to.
- At the end, download the Notebook (using File > Download > Download .ipynb). Once it has completed please zip the file up. This is the artefact you would submit for the in-class test.
- You can also save a copy to your GitHub using the instructions from my.wbs (1b asychronous tasks).

**Suggested timing**

| Minutes | Focus |
| --- | --- |
| 0 to 10 | Look at the tables and note what you find |
| 10 to 25 | Clean and prepare the data |
| 25 to 38 | Join and aggregate to answer the question |
| 38 to 45 | Check your answer and write up |

At the end, use **Runtime, then Restart session and run all** and note anything that fails.


## The data

A fictional bike-hire scheme: three tables covering trips taken in March 2025.

| Table | What it holds |
| --- | --- |
| `trips` | One record per trip: bike, start and end station, start time, duration and user type |
| `bikes` | The bikes, with type and purchase year |
| `stations` | The docking stations, with zone and capacity |

**How they connect:** `trips.bike_id` to `bikes.bike_id`, and `trips.start_station_id` (and `end_station_id`) to `stations.station_id`.

Check the real column names in the next cell before you write anything.


In [ ]:
# ---- Setup: run this cell first ----
!pip install -q duckdb

import glob
import os

import duckdb
import pandas as pd

DATA_URL = "https://github.com/MJMortensonWarwick/adminaeaiv/releases/download/dbsession/bike-hire-practice.zip"

DATA_DIR = "/content/bikes"
# Download only if no CSV files are there yet. The CSVs may sit inside a folder
# within the zip; the search below looks in sub-folders too.
if DATA_URL and not glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True):
    import urllib.request
    import zipfile
    os.makedirs(DATA_DIR, exist_ok=True)
    urllib.request.urlretrieve(DATA_URL, "/content/bikes.zip")
    zipfile.ZipFile("/content/bikes.zip").extractall(DATA_DIR)

FILES = {"trips": "trips.csv", "bikes": "bikes.csv", "stations": "stations.csv"}

tables, missing = {}, []
for name, filename in FILES.items():
    found = glob.glob(os.path.join(DATA_DIR, "**", filename), recursive=True)
    if found:
        tables[name] = pd.read_csv(found[0])
    else:
        missing.append(filename)

if missing:
    found_csvs = sorted(os.path.basename(f) for f in glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True))
    raise FileNotFoundError(
        f"Missing files: {missing}. CSV files found under {DATA_DIR}: {found_csvs}. Ask for help."
    )

# Make every table available as a DuckDB table, so you can use SQL or pandas.
con = duckdb.connect()
for name, df in tables.items():
    con.register(name, df)


def q(sql):
    """Run SQL against the tables above and return a pandas DataFrame."""
    return con.sql(sql).df()


for name, df in tables.items():
    print(f"{name}: {df.shape[0]:,} rows, {df.shape[1]} columns")
    print("   ", list(df.columns))

display(tables["trips"])
display(tables["bikes"])
display(tables["stations"])


trips: 612 rows, 7 columns
    ['trip_id', 'bike_id', 'start_station_id', 'end_station_id', 'start_time', 'duration_min', 'user_type']
bikes: 60 rows, 3 columns
    ['bike_id', 'bike_type', 'purchase_year']
stations: 12 rows, 4 columns
    ['station_id', 'station_name', 'zone', 'capacity']


,trip_id,bike_id,start_station_id,end_station_id,start_time,duration_min,user_type
0,T0001,B028,S05,S01,2025-03-01 05:52:00,21.0,Casual
1,T0002,B043,S12,S12,2025-03-01 06:31:00,27.0,CASUAL
2,T0003,B045,S11,S07,2025-03-01 07:06:00,18.0,Casual
3,T0004,B051,S04,S04,2025-03-01 07:06:00,34.0,Member
4,T0005,B049,S01,S09,2025-03-01 07:36:00,16.0,Member
...,...,...,...,...,...,...,...
607,T0596,B051,S02,S02,2025-03-31 13:51:00,NaN,Casual
608,T0597,B011,S01,S04,2025-03-31 14:00:00,19.0,Member
609,T0598,B046,S06,S11,2025-03-31 15:38:00,12.0,Member
610,T0599,B027,S04,S02,31/03/2025 18:38,21.0,casual


,bike_id,bike_type,purchase_year
0,B001,Standard,2021
1,B002,Electric,2022
2,B003,electric,2024
3,B004,Standard,2021
4,B005,STANDARD,2022
5,B006,Standard,2024
6,B007,standard,2024
7,B008,Standard,2021
8,B009,Standard,2021
9,B010,Standard,2023


,station_id,station_name,zone,capacity
0,S01,Market Square,Centre,29.0
1,S02,Railway Station,Centre,23.0
2,S03,Canal Basin,North,24.0
3,S04,University Gate,West,28.0
4,S05,Park Road,South,22.0
5,S06,Library Corner,Centre,26.0
6,S07,Hospital Entrance,East,27.0
7,S08,Riverside Walk,South,14.0
8,S09,Old Town Hall,Centre,NaN
9,S10,Retail Park,North,16.0


In [ ]:
from google.colab import sheets
sheet = sheets.InteractiveSheet(df=tables["trips"])

https://docs.google.com/spreadsheets/d/1-I-lhYsh47ZvlMdXKRtgN9TjCKGtfJVu8FPg-5z2v8s/edit#gid=0


In [ ]:
from google.colab import sheets
sheet = sheets.InteractiveSheet(df=tables["bikes"])

https://docs.google.com/spreadsheets/d/1sglZi5ggwE1H6axEL7ZIQ9swA5u-1oBghS6PkB4iIjA/edit#gid=0


In [ ]:
from google.colab import sheets
sheet = sheets.InteractiveSheet(df=tables["stations"])

https://docs.google.com/spreadsheets/d/1zimE2ZGeVtuPZE9CayOIsaqfSSJO5LKRP98jFZxvM0A/edit#gid=0


## Examples: two ways to work with the tables

You can use SQL, pandas, or a mix of both. The next two cells show each. Run them and change them to see what happens. You do not have to use them in your answer.


In [ ]:
# Example 1: SQL
# q() runs SQL against any of the tables above and returns a pandas DataFrame.
from IPython.display import display

display(q("SELECT * FROM trips LIMIT 5"))

display(q("SELECT COUNT(*) AS n_rows, COUNT(DISTINCT trip_id) AS n_distinct_trips FROM trips"))

display(q("""
    SELECT t.trip_id, s.station_name
    FROM trips AS t
    JOIN stations AS s ON t.start_station_id = s.station_id
    LIMIT 5
"""))


,trip_id,bike_id,start_station_id,end_station_id,start_time,duration_min,user_type
0,T0001,B028,S05,S01,2025-03-01 05:52:00,21.0,Casual
1,T0002,B043,S12,S12,2025-03-01 06:31:00,27.0,CASUAL
2,T0003,B045,S11,S07,2025-03-01 07:06:00,18.0,Casual
3,T0004,B051,S04,S04,2025-03-01 07:06:00,34.0,Member
4,T0005,B049,S01,S09,2025-03-01 07:36:00,16.0,Member


,n_rows,n_distinct_trips
0,612,600


,trip_id,station_name
0,T0001,Park Road
1,T0002,Cathedral Close
2,T0003,Sports Centre
3,T0004,University Gate
4,T0005,Market Square


In [ ]:
# Example 2: pandas
# Each table is also a pandas DataFrame, stored in the dictionary `tables`.
from IPython.display import display

trips = tables["trips"]
display(trips.head())
print(trips.shape)
print(trips.dtypes)

joined = tables["trips"].merge(tables["stations"], left_on="start_station_id", right_on="station_id", how="inner")
display(joined[["trip_id", "station_name"]].head())

# You can mix the two: run SQL, then use pandas methods on the result
result = q("SELECT start_station_id FROM trips")
print(result["start_station_id"].nunique(), "different start stations in trips")


,trip_id,bike_id,start_station_id,end_station_id,start_time,duration_min,user_type
0,T0001,B028,S05,S01,2025-03-01 05:52:00,21.0,Casual
1,T0002,B043,S12,S12,2025-03-01 06:31:00,27.0,CASUAL
2,T0003,B045,S11,S07,2025-03-01 07:06:00,18.0,Casual
3,T0004,B051,S04,S04,2025-03-01 07:06:00,34.0,Member
4,T0005,B049,S01,S09,2025-03-01 07:36:00,16.0,Member


(612, 7)
trip_id              object
bike_id              object
start_station_id     object
end_station_id       object
start_time           object
duration_min        float64
user_type            object
dtype: object


,trip_id,station_name
0,T0001,Park Road
1,T0002,Cathedral Close
2,T0003,Sports Centre
3,T0004,University Gate
4,T0005,Market Square


12 different start stations in trips


## Your brief

**From:** the scheme manager

> We want to know whether electric bikes are used for longer trips than standard bikes, and whether that differs by zone. Please give us the **average trip duration by bike type and by the zone of the start station**.

![](https://drive.google.com/uc?export=view&id=1eH5Q3l4LnzwZ-XcT_kJDeHoKA2rufiOW)

**What your final table should look like**

One row per combination of bike type and zone, with the number of trips and the average duration in minutes. State what one row represents.

**Things to think about** (you do not have to solve all of them)

- Are there **missing**, **duplicated** or **odd** values in the tables you use? What will you do about each, and why?
- Do the bike **types** and **dates** look consistent?
- How will you check your final table is believable?


## 1. Look at the data

*What does one row of each table represent? Which columns will you use? What looks odd, missing or inconsistent? Write down what you find.*

#### 1.1 What does one row of each table represent?

### Table: `trips`

**Rows:** 612 | **Columns:** 7 | **One row =** one trip

| Column | Type | Description | Missing |
|---|---|---|---|
| `trip_id` | str | Unique trip identifier | 0 (0.0%) |
| `bike_id` | str | Bike used  | 0 (0.0%) |
| `start_station_id` | str | Station where trip started | 0 (0.0%) |
| `end_station_id` | str | Station where trip ended | 0 (0.0%) |
| `start_time` | str | Trip start timestamp  | 0 (0.0%) |
| `duration_min` | float64 | Trip duration in minutes | 30 (4.9%) |
| `user_type` | str | Member or Casual (inconsistent case in raw data) | 0 (0.0%) |

### Table: `bikes`

**Rows:** 60 | **Columns:** 3 | **One row =** one bike

| Column | Type | Description | Missing |
|---|---|---|---|
| `bike_id` | str | Unique bike identifier | 0 (0.0%) |
| `bike_type` | str | Standard or Electric (inconsistent spelling in raw data) | 0 (0.0%) |
| `purchase_year` | int64 | Year the bike was purchased | 0 (0.0%) |

### Table: `stations`

**Rows:** 12 | **Columns:** 4 | **One row =** one docking station

| Column | Type | Description | Missing |
|---|---|---|---|
| `station_id` | str | Unique station identifier | 0 (0.0%) |
| `station_name` | str | Station name | 0 (0.0%) |
| `zone` | str | Zone of the station | 0 (0.0%) |
| `capacity` | float64 | Number of docking points | 1 (8.3%) |

### 1.2 Which columns will you use?
| Table | Column | Used? | Why |
|---|---|---|---|
| `trips` | `trip_id` | Yes | Count trips, detect duplicates |
| `trips` | `bike_id` | Yes | Join to `bikes` |
| `trips` | `start_station_id` | Yes | Join to `stations` (start station zone) |
| `trips` | `duration_min` | Yes | Metric: average trip duration |
| `trips` | `start_time` | Yes | Filter for March 2025 |
| `trips` | `end_station_id` | No | Only start station zone required |
| `trips` | `user_type` | No | Irrelevant to this question |
| `bikes` | `bike_id` | Yes | Join to `trips` |
| `bikes` | `bike_type` | Yes | Group: Standard / Electric |
| `bikes` | `purchase_year` | No | Not required |
| `stations` | `station_id` | Yes | Join to `start_station_id` |
| `stations` | `zone` | Yes | Group by start station zone |
| `stations` | `station_name` | No | Zone is sufficient |
| `stations` | `capacity` | No | Not required, also has missing data |


### 1.3 What looks odd, missing or inconsistent? Write down what you find.
### Findings
| # | Table | Column | Issue | Rows affected | Impact for analysis |
|---|---|---|---|---|---|
|1|`trips`|all|Duplicate trip records|12|Overcounts trips and biases average duration|
|2|`trips`|`duration_min`|Missing values|30 (4.9%)|Cannot calculate average duration for these trips|
|3|`trips`|`duration_min`|Extreme long durations (24–48h)|3|Outliers that will inflate the mean|
|4|`trips`|`start_time`|Mixed text date formats|98|Cannot filter dates reliably before parsing|
|5|`trips`|`user_type`|Inconsistent case + trailing spaces|many|Not required for this task|
|6|`bikes`|`bike_type`|Inconsistent spelling/case (E-bike variant)|60|Critical: will create multiple unwanted groups|
|7|`stations`|`capacity`|1 missing value|1|Not used in this analysis|

### Checks that were fine
- No orphan IDs; all `bike_id` / `start_station_id` in trips exist in reference tables
- Primary keys unique, joins will not duplicate rows
- All parsed dates are within March 2025
- Join keys have no leading/trailing spaces

### Assumptions
- `E-bike` = Electric bike
- `dd/mm/yyyy` dates are day-first format (March 2025)
- Trips >4 hours are data errors and not valid rides


## 2. Clean and prepare

*Decide what to do about each problem you found, say why, and do it. Short cells that you can re-run are better than one long cell.*

#### Keep raw source tables unchanged + make working copies.

In [ ]:
trips_clean = tables["trips"].copy()
bikes_clean = tables["bikes"].copy()
stations_clean = tables["stations"].copy()

### Action 1: Remove duplicate trips
Problem: 12 duplicated trip records with identical trip_id.



In [ ]:
rows_before = len(trips_clean)
# Keep only the first row for each trip_id
trips_clean = trips_clean.drop_duplicates(subset="trip_id", keep="first")
print("Duplicate trip records removed:", rows_before - len(trips_clean))
print("Remaining rows after Action1:", len(trips_clean))

# Verify no duplicate trip_id left
print("Duplicate trip IDs remaining:", trips_clean["trip_id"].duplicated().sum())

Duplicate trip records removed: 12
Remaining rows after Action1: 600
Duplicate trip IDs remaining: 0


### Action 2: Remove rows with missing duration_min
Problem: 30 trips have NULL duration_min.


In [ ]:
rows_before = len(trips_clean)
trips_clean = trips_clean.dropna(subset=["duration_min"])
print("Rows removed (missing duration_min):", rows_before - len(trips_clean))
print("Remaining rows after Action2:", len(trips_clean))



Rows removed (missing duration_min): 30
Remaining rows after Action2: 570


### Action4: Parse start_time into standard datetime
Problem: start_time stored as text with two different date formats.


In [ ]:
import pandas as pd
rows_before = len(trips_clean)

trips_clean["start_time_ts"] = pd.to_datetime(trips_clean["start_time"], dayfirst=True, errors="coerce")

start_date = pd.Timestamp("2025-03-01")
end_date = pd.Timestamp("2025-03-31 23:59:59")
trips_clean = trips_clean[(trips_clean["start_time_ts"] >= start_date) & (trips_clean["start_time_ts"] <= end_date)]

print("Rows removed during date parsing & filter:", rows_before - len(trips_clean))
print("Remaining rows after Action4:", len(trips_clean))
print("Date range: ", trips_clean["start_time_ts"].min(), "to", trips_clean["start_time_ts"].max())



Rows removed during date parsing & filter: 564
Remaining rows after Action4: 6
Date range:  2025-03-03 07:32:00 to 2025-03-03 19:29:00


### Action5: Clean user_type (not used in analysis)
Problem: inconsistent case and trailing spaces in user_type.

In [ ]:
rows_before = len(trips_clean)
trips_clean["user_type_clean"] = trips_clean["user_type"].str.strip().str.upper()
print("No rows removed in Action5, rows remain:", len(trips_clean))
print("Distinct cleaned user_type values:")
print(trips_clean["user_type_clean"].unique())


No rows removed in Action5, rows remain: 6
Distinct cleaned user_type values:
['MEMBER' 'CASUAL']


### Action6: Normalise bike_type (critical grouping column)
Problem: bike_type has inconsistent spelling, case, spaces, and "E-bike".

In [ ]:
# Remove whitespace and convert to uppercase for matching
bikes_clean["bike_type_clean"] = bikes_clean["bike_type"].str.strip().str.upper()

# 映射：E-BIKE / ELECTRIC → Electric；其余全部 Standard
def normalise_type(raw_text):
    if raw_text in ["ELECTRIC", "E-BIKE"]:
        return "Electric"
    else:
        return "Standard"

bikes_clean["bike_type_clean"] = bikes_clean["bike_type_clean"].apply(normalise_type)

print("Distinct cleaned bike types:")
print(bikes_clean["bike_type_clean"].unique())
print("Trips dataframe unchanged, trips still =", len(trips_clean))




Distinct cleaned bike types:
['Standard' 'Electric']
Trips dataframe unchanged, trips still = 6


### Join Table


In [ ]:
# Join trips with bikes
trips_with_bike = pd.merge(
    trips_clean,
    bikes_clean[["bike_id", "bike_type_clean"]],
    on="bike_id",
    how="inner"
)

# Join result with stations_clean to get zone
final_joined = pd.merge(
    trips_with_bike,
    stations_clean[["station_id", "zone"]],
    left_on="start_station_id",
    right_on="station_id",
    how="inner"
)

print("Total rows after final inner join:", len(final_joined))


Total rows after final inner join: 6


## 4. Check

*How do you know the answer is right? Compare a count, spot check a few records, or recalculate one number a different way. What did you find?*

### Check1: Compare total trip count
Compare total trips in joined table against the sum of trips in the final grouped table.
If they match, grouping did not lose or duplicate rows.


In [ ]:
# Total trips in joined raw data
total_raw = len(final_joined)
print("Total trips in joined dataset:", total_raw)

# Group to get count per zone + bike type
group_result = final_joined.groupby(["zone", "bike_type_clean"])["duration_min"].agg(
    trip_count = "count",
    avg_duration_min = "mean"
).reset_index()

print("\nGrouped summary table:")
print(group_result)

# Sum all trip counts from grouped table
sum_grouped_counts = group_result["trip_count"].sum()
print("\nSum of trips from grouped table:", sum_grouped_counts)

if total_raw == sum_grouped_counts:
    print("✅ Check passed: Counts match, no missing or duplicated rows after grouping.")
else:
    print("❌ Check failed: Count mismatch.")


Total trips in joined dataset: 6

Grouped summary table:
     zone bike_type_clean  trip_count  avg_duration_min
0  Centre        Electric           2              24.0
1  Centre        Standard           1              18.0
2   South        Electric           1               8.0
3   South        Standard           1              23.0
4    West        Standard           1              20.0

Sum of trips from grouped table: 6
✅ Check passed: Counts match, no missing or duplicated rows after grouping.


### Check2: Spot check individual records
Pick one zone + bike type group, look at raw underlying rows to confirm data matches the group.


In [ ]:
# Pick one group from the result table, for example zone S01, Electric
sample_zone = "S01"
sample_type = "Electric"

sample_rows = final_joined[
    (final_joined["zone"] == sample_zone) &
    (final_joined["bike_type_clean"] == sample_type)
][["zone", "bike_type_clean", "duration_min"]]

print(f"Raw records for zone {sample_zone}, {sample_type}:")
print(sample_rows)
print("Number of raw rows in this group:", len(sample_rows))
print("Manual average of this group:", sample_rows["duration_min"].mean())


Raw records for zone S01, Electric:
Empty DataFrame
Columns: [zone, bike_type_clean, duration_min]
Index: []
Number of raw rows in this group: 0
Manual average of this group: nan


### Check3: Recalculate average using an alternative method
Use pandas pivot_table to recompute the average duration and trip count, compare with groupby result.


In [ ]:
# Select only the columns required for this question.
bikes_for_join = bikes_clean[
    ["bike_id", "bike_type_clean"]
].rename(columns={"bike_type_clean": "bike_type"})

stations_for_join = stations_clean[
    ["station_id", "zone"]
]

# Join trips to bikes, then to the START station.
joined = trips_clean.merge(
    bikes_for_join,
    on="bike_id",
    how="left",
    validate="many_to_one",
    indicator="bike_match"
)

joined = joined.merge(
    stations_for_join,
    left_on="start_station_id",
    right_on="station_id",
    how="left",
    validate="many_to_one",
    indicator="station_match"
)

print("Rows before joins:", len(trips_clean))
print("Rows after joins:", len(joined))
print("Unmatched bikes:",
      int((joined["bike_match"] == "left_only").sum()))
print("Unmatched start stations:",
      int((joined["station_match"] == "left_only").sum()))

# Keep rows with all fields needed for the answer.
analysis_data = joined.dropna(
    subset=["bike_type", "zone", "duration_min"]
).copy()

# One output row per bike type and start-station zone.
final_table = (
    analysis_data
    .groupby(["bike_type", "zone"], as_index=False)
    .agg(
        trip_count=("trip_id", "nunique"),
        average_duration_minutes=("duration_min", "mean")
    )
    .sort_values(["bike_type", "zone"])
)

final_table["average_duration_minutes"] = (
    final_table["average_duration_minutes"].round(2)
)

display(final_table)

Rows before joins: 6
Rows after joins: 6
Unmatched bikes: 0
Unmatched start stations: 0


,bike_type,zone,trip_count,average_duration_minutes
0,Electric,Centre,2,24.0
1,Electric,South,1,8.0
2,Standard,Centre,1,18.0
3,Standard,South,1,23.0
4,Standard,West,1,20.0


In [ ]:
from google.colab import sheets
sheet = sheets.InteractiveSheet(df=final_table)

https://docs.google.com/spreadsheets/d/1qtLxpt65iriHZtlerAqLp4xauKqI47yxWFdsL5wyZEY/edit#gid=0


## 5. Prompts and AI notes

*Paste each prompt as you sent it. Note what you accepted, changed or rejected and why.*

## 6. What I could not verify

*What remains unchecked? What would you do next?*